In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

full_data = pd.read_excel(r'', sheet_name=None)

foodsel_sheet = pd.DataFrame(full_data['foodsel_'])


In [2]:
#Here i used shorter names for columns, so that the code is easier to understand the following are the names used.
Quantity = '3а Охирги давр ичида уй хўжалигингиз аъзолари томонидан сотиб олинган _${fooditemlabel}_нинг умумий миқдорини аниқланг.'
Price_UZS = '3б Охирги давр ичида уй хўжалигингиз аъзолари томонидан сотиб олинган _${fooditemlabel}_нинг умумий қийматини аниқланг (сўмда).'
Time_duration = '3с _${fooditemlabel}_ни сотиб олинган даврини танланг'
Purchased_location = '4. Охирги давр ичида _${fooditemlabel}_нинг сотиб олинган жойи:'
HouseholdID = 'm01uuid'

#Here I have seperated each and every unit of measurment into another variable.
#First I did for the first sheet foodsel_ others have other units for now these 3 would be enough.
items_кг = foodsel_sheet['fooditemlabel'].str.contains(r'\(кг\)', case=False, na=False)
items_дона = foodsel_sheet['fooditemlabel'].str.contains(r'\(дона\)', case=False, na=False)
items_литр = foodsel_sheet['fooditemlabel'].str.contains(r'\(литр\)', case=False, na=False)
items_гр = foodsel_sheet['fooditemlabel'].str.contains(r'\(гр\)', case=False, na=False)
items_кишилик = foodsel_sheet['fooditemlabel'].str.contains('кишилик', case=False, na=False)
items_қути = foodsel_sheet['fooditemlabel'].str.contains(r'\(қути\)', case=False, na=False)
items_порция = foodsel_sheet['fooditemlabel'].str.contains('порция', case=False, na=False)
# verify that we got all the rows

all_items = (items_кг | items_дона | items_литр | items_гр | items_кишилик | items_қути | items_порция)
print(f"\nTotal covered: {all_items.sum()} rows")

#Saving all files into excel 
# IF THIS IS RUN ALREADY NO NEED TO RUN
filtered_kg = foodsel_sheet[items_кг]
filtered_kg.to_excel('Items_in_kg.xlsx', index=False)

filtered_gr = foodsel_sheet[items_гр]
filtered_gr.to_excel('Items_in_grams.xlsx', index=False)

filtered_pieces = foodsel_sheet[items_дона]
filtered_pieces.to_excel('Items_in_peices.xlsx', index=False)

filtered_litr = foodsel_sheet[items_литр]
filtered_litr.to_excel('Items_in_litr.xlsx', index=False)

filtered_person = foodsel_sheet[items_кишилик]
filtered_person.to_excel('Items_person.xlsx', index=False)

filtered_box = foodsel_sheet[items_қути]
filtered_box.to_excel('Items_box.xlsx', index=False)

filtered_portion = foodsel_sheet[items_порция]
filtered_portion.to_excel('Items_portion.xlsx', index=False)



Total covered: 66856 rows


In [4]:

purchase_locations = [
    "Бозор, павилон,Киоскадан",
    "Дўкондан",
    "Давлатдан",
    "Жисмоний шахсдан",
    "Интернет-дўконлар орқали",
    "Юридик шахсдан (ЯТТ, МЧЖ)"
]

duration_map = {
    "7 кун": 7/30,
    "1 ой" : 1,
    "3 ой" : 3,
    "6 ой" : 6,
    "1 йил": 12
}

min_qty_floor = {
    'гр'      : 1,
    'кг'      : 0.1,
    'литр'    : 0.1,
    'дона'    : 1,
    'қути'    : 1,
    'порция'  : 1,
    'кишилик' : 1,
}

# unit name → (boolean mask, unit_type string)
unit_data = {
    'кг'      : (items_кг,       'кг'),
    'гр'      : (items_гр,       'гр'),
    'дона'    : (items_дона,     'дона'),
    'литр'    : (items_литр,     'литр'),
    'кишилик' : (items_кишилик,  'кишилик'),
    'қути'    : (items_қути,     'қути'),
    'порция'  : (items_порция,   'порция'),
}

# ── HELPER FUNCTIONS ───────────────────────────────────────────────────────

def iqr_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = max(Q1 - 1.5 * IQR, 0)   # floor at 0, price/qty can't be negative
    upper = Q3 + 1.5 * IQR
    return lower, upper

def apply_iqr_flag(df, column):
    if df['fooditemlabel'].nunique() == 1:
        lower, upper = iqr_bounds(df[column])
        return (df[column] < lower) | (df[column] > upper)
    else:
        def flag_group(group):
            lower, upper = iqr_bounds(group[column])
            return (group[column] < lower) | (group[column] > upper)
        return df.groupby('fooditemlabel', group_keys=False).apply(flag_group)

# ── MAIN FUNCTION ──────────────────────────────────────────────────────────

def flag_inconsistencies(df, unit_type):
    df = df.copy()
    df[Purchased_location] = df[Purchased_location].str.strip()
    df[Time_duration]      = df[Time_duration].str.strip()
    df['duration_months']  = df[Time_duration].map(duration_map)

    # CHECK 1: Zero price with store purchase
    mask_zero_price = (
        df[Purchased_location].isin(purchase_locations)
    ) & (
        df[Price_UZS] == 0
    )

    # CHECK 2: Unit price outlier (IQR per product)
    df['unit_price'] = df[Price_UZS] / df[Quantity]
    mask_unit_price_outlier = apply_iqr_flag(df, 'unit_price')

    # CHECK 3: Quantity range outlier (IQR per product)
    mask_qty_range_outlier = apply_iqr_flag(df, Quantity)

    # CHECK 4: Monthly quantity outlier (IQR per product)
    df['qty_per_month'] = df[Quantity] / df['duration_months']
    mask_qty_outlier = apply_iqr_flag(df, 'qty_per_month')

    # CHECK 5: Duplicate household + product
    mask_duplicate = df.duplicated(subset=[HouseholdID, 'fooditemlabel'], keep=False)

    # CHECK 6: Suspiciously round numbers
    round_numbers = [1, 10, 100, 1000, 10000]
    mask_round = df[Quantity].isin(round_numbers) & df[Price_UZS].isin(round_numbers)

    # CHECK 7: Fractional quantity (only for whole-unit types)
    whole_units = ['дона', 'қути', 'кишилик', 'порция']
    if unit_type in whole_units:
        mask_fraction = df[Quantity] != df[Quantity].round()
    else:
        mask_fraction = pd.Series(False, index=df.index)

    # CHECK 8: Implausibly small quantity (absolute floor per unit)
    floor = min_qty_floor.get(unit_type, 0)
    mask_implausible_qty = df[Quantity] < floor

    # CHECK 9: Missing values in key columns
    mask_missing = (
        df[Quantity].isna() |
        df[Price_UZS].isna() |
        df[Time_duration].isna() |
        df[Purchased_location].isna()
    )

    # COMBINE the results of the flagged values. First we make new columns for the flagged values
    df['flag_zero_price']         = mask_zero_price
    df['flag_unit_price_outlier'] = mask_unit_price_outlier
    df['qty_range_outlier']       = mask_qty_range_outlier
    df['flag_qty_outlier']        = mask_qty_outlier
    df['flag_duplicate']          = mask_duplicate
    df['flag_round_numbers']      = mask_round
    df['flag_fraction_qty']       = mask_fraction
    df['flag_implausible_qty']    = mask_implausible_qty
    df['flag_missing_values'] = mask_missing

    # Now we take all the flagged values and then by using any(axis=1) looks through each row, horizontally and checks if 
    # any of the values are True if the rows have at least 1 true then all is considered and flagged for review. If False
    # Then there were no errors and that row is dropped.

    all_flags = [
    'flag_zero_price', 'flag_unit_price_outlier', 'qty_range_outlier',
    'flag_qty_outlier', 'flag_duplicate', 'flag_round_numbers',
    'flag_fraction_qty', 'flag_implausible_qty', 'flag_missing_values'
]
    any_flag   = df[all_flags].any(axis=1)
    flagged_df = df[any_flag]

    # SUMMARY THIS IS RUN IF YOU NEED A SUMMARY OF HOW MANY FLAGGED VALUES HAVE BEEN FOUND. 
    # print(f"\n{'='*50}")
    # print(f"  Unit: {unit_type} | Total rows: {len(df)}")
    # print(f"  flag_zero_price:          {mask_zero_price.sum()}")
    # print(f"  flag_unit_price_outlier:  {mask_unit_price_outlier.sum()}")
    # print(f"  qty_range_outlier:        {mask_qty_range_outlier.sum()}")
    # print(f"  flag_qty_outlier:         {mask_qty_outlier.sum()}")
    # print(f"  flag_duplicate:           {mask_duplicate.sum()}")
    # print(f"  flag_round_numbers:       {mask_round.sum()}")
    # print(f"  flag_fraction_qty:        {mask_fraction.sum()}")
    # print(f"  flag_implausible_qty:     {mask_implausible_qty.sum()}")
    # print(f"  Total flagged rows:       {any_flag.sum()}")
    # print(f"  flag_missing_values:      {mask_missing.sum()}")

    return flagged_df, df  # flagged rows + full df with flag columns

# RUN LOOP 

for unit_name, (mask, unit_type) in unit_data.items():
    filtered_df = foodsel_sheet[mask]
    
    flagged, full = flag_inconsistencies(filtered_df, unit_type)
    
    flagged.to_excel(f'{unit_name}_маҳсулотлар.xlsx', index=False)


# RUN THIS IF THE FLAGGED VALUES ACTUALLY ARE CONSISTENT, MAIN PURPOSE IS FOR MANUALL CHECK.
# thresholds = full.groupby('fooditemlabel')['unit_price'].apply(
#         lambda x: pd.Series({
#             'Q1'    : x.quantile(0.25),
#             'Q3'    : x.quantile(0.75),
#             'IQR'   : x.quantile(0.75) - x.quantile(0.25),
#             'lower' : x.quantile(0.25) - 1.5 * (x.quantile(0.75) - x.quantile(0.25)),
#             'upper' : x.quantile(0.75) + 1.5 * (x.quantile(0.75) - x.quantile(0.25)),
#             'median': x.median(),
#             'min'   : x.min(),
#             'max'   : x.max()
#         })
#     ).reset_index()

# thresholds.to_excel(f'thresholds_{unit_name}.xlsx', index=False)